**CI twin of `ch03-cost-gradient-descent.qmd`.** Generated by `infra/ci/make_twin.py` (P1-D10); do not edit by hand. It runs the chapter's worked example and exercise solutions under CPython so the R10 gate proves they work (DECISIONS D0010).

In [ ]:
# Make the single-sourced grader importable under CPython (no paste; P1-D9).
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'lib' / 'grader.py').exists():
        sys.path.insert(0, str(_p))
        break
from lib.grader import run_tests

In [ ]:
from lib.data import load_csv
import matplotlib.pyplot as plt

homes = load_csv("california-housing-sample")
x = homes["MedInc"].to_list()
y = homes["MedHouseVal"].to_list()
n = len(x)

def mse(w, b):
    return sum(((w * xi + b) - yi) ** 2 for xi, yi in zip(x, y)) / n

ws = [i / 100 for i in range(0, 91)]
costs = [mse(w, 0.410) for w in ws]

fig, ax = plt.subplots(figsize=(5, 3.2))
ax.plot(ws, costs)
ax.set_xlabel("weight w  (with b fixed at 0.410)")
ax.set_ylabel("MSE")
plt.show()

In [ ]:
def gradient_descent(x, y, lr, steps):
    w, b = 0.0, 0.0
    for step in range(1, steps + 1):
        errors = [(w * xi + b) - yi for xi, yi in zip(x, y)]
        grad_w = 2 * sum(e * xi for e, xi in zip(errors, x)) / n
        grad_b = 2 * sum(errors) / n
        w = w - lr * grad_w
        b = b - lr * grad_b
        if step in (1, 10, 100, 1000, 3000):
            print(f"step {step:5}: w={w:.3f}  b={b:.3f}  cost={mse(w, b):.3f}")
    return w, b

w, b = gradient_descent(x, y, lr=0.01, steps=3000)
print(f"\nwalked to:    w={w:.3f}, b={b:.3f}")
print("Ch2 formula:  w=0.432, b=0.410")

In [ ]:
for lr, label in [(0.06, "too high"), (0.001, "too low")]:
    w, b = 0.0, 0.0
    trace = []
    for step in range(6):
        errors = [(w * xi + b) - yi for xi, yi in zip(x, y)]
        w -= lr * 2 * sum(e * xi for e, xi in zip(errors, x)) / n
        b -= lr * 2 * sum(errors) / n
        trace.append(round(mse(w, b), 3))
    print(f"lr={lr} ({label}): cost per step = {trace}")

In [ ]:
xs = [1.0, 2.0, 3.0]
ys = [2.0, 4.0, 6.0]
w, b, lr = 0.0, 0.0, 0.1

for _ in range(3):
    errors = [(w * xi + b) - yi for xi, yi in zip(xs, ys)]
    grad_w = 2 * sum(e * xi for e, xi in zip(errors, xs)) / len(xs)
    grad_b = 2 * sum(errors) / len(xs)
    w = w - lr * grad_w
    b = b - lr * grad_b

run_tests([
    ("w after 3 steps", round(w, 4), 1.7007),
    ("b after 3 steps", round(b, 4), 0.6862),
])

In [ ]:
def mse(y_true, y_pred):
    return sum((t - p) ** 2 for t, p in zip(y_true, y_pred)) / len(y_true)

def one_step(w, b, xs, ys, lr):
    errors = [(w * xi + b) - yi for xi, yi in zip(xs, ys)]
    grad_w = 2 * sum(e * xi for e, xi in zip(errors, xs)) / len(xs)
    grad_b = 2 * sum(errors) / len(xs)
    return w - lr * grad_w, b - lr * grad_b

run_tests([
    ("mse mixed gaps", mse([2.0, 3.0, 1.0], [1.5, 3.5, 2.0]), 0.5),
    ("mse perfect", mse([1.0, 4.0], [1.0, 4.0]), 0.0),
    ("first step w", one_step(0.0, 0.0, [1.0, 2.0, 3.0], [2.0, 4.0, 6.0], 0.1)[0],
     28 / 15),
    ("first step b", one_step(0.0, 0.0, [1.0, 2.0, 3.0], [2.0, 4.0, 6.0], 0.1)[1],
     0.8),
], tol=1e-9)